# 实验2 - 3.2 螺旋数据分类（Spiral Classification）

对应实验指导：[实验2：螺旋数据分类](https://oucaigroup.feishu.cn/wiki/Nbddwe54fiolLFkoslfcdYcAnZe)

用神经网络对三类螺旋分布的数据进行分类，对比**线性模型**与**加入 ReLU 激活函数的两层神经网络**的分类效果。

### 0. 准备工作：引入绘图库（plot_lib.py 已与 notebook 放在同一目录）

In [ ]:
import random
import math

import torch
from torch import nn, optim
from IPython import display

from plot_lib import plot_data, plot_model, set_default

# 本地无 GPU 时自动回退到 CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('device: ', device)

In [ ]:
# 设置随机数种子，保证结果可复现
seed = 12345
random.seed(seed)
torch.manual_seed(seed)

N = 1000   # 每类样本数量
D = 2      # 每个样本的特征维度
C = 3      # 样本类别数
H = 100    # 神经网络隐层单元数量

### 1. 初始化 X（特征矩阵）和 Y（标签）

In [ ]:
X = torch.zeros(N * C, D).to(device)
Y = torch.zeros(N * C, dtype=torch.long).to(device)
for c in range(C):
    index = 0
    t = torch.linspace(0, 1, N)   # 在 [0, 1] 间均匀取 N 个数
    # 三类样本的角度区间相互交错并加入噪声，构成螺旋形分布
    inner_var = torch.linspace((2 * math.pi / C) * c,
                               (2 * math.pi / C) * (2 + c), N) + torch.randn(N) * 0.2
    for ix in range(N * c, N * (c + 1)):
        X[ix] = t[index] * torch.FloatTensor(
            (math.sin(inner_var[index]), math.cos(inner_var[index])))
        Y[ix] = c
        index += 1

print("Shapes:")
print("X:", X.size())   # [3000, 2]：3000 个样本，每个 2 维特征
print("Y:", Y.size())   # [3000]：每个样本的类别 0/1/2

In [ ]:
# 可视化训练数据（三类螺旋）
set_default()
plot_data(X, Y)

### 2. 构建线性模型分类（两层线性层，无激活函数）

In [ ]:
learning_rate = 1e-3
lambda_l2 = 1e-5

model = nn.Sequential(
    nn.Linear(D, H),
    nn.Linear(H, C)
)
model.to(device)

# 交叉熵损失
criterion = torch.nn.CrossEntropyLoss()
# 随机梯度下降优化器（带 L2 正则）
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=lambda_l2)

for t in range(1000):
    y_pred = model(X)                       # 前向传播，得到预测
    loss = criterion(y_pred, Y)             # 计算损失
    score, predicted = torch.max(y_pred, 1) # 每个样本取得分最大的类别
    acc = (Y == predicted).sum().float() / len(Y)
    print('[EPOCH]: %i, [LOSS]: %.6f, [ACCURACY]: %.3f' % (t, loss.item(), acc))
    display.clear_output(wait=True)

    optimizer.zero_grad()                   # 反向传播前把梯度清零
    loss.backward()                         # 反向传播计算梯度
    optimizer.step()                        # 更新参数

In [ ]:
# 查看预测结果的结构与含义
print('y_pred.shape:', y_pred.shape)      # [3000, 3]，每个样本有 3 个类别的得分
print('y_pred[10, :]:', y_pred[10, :])    # 第 10 个样本的 3 个得分
print('score[10]:', score[10])            # 其中的最大值
print('predicted[10]:', predicted[10])    # 最大值所在位置（预测类别）

In [ ]:
# 绘制训练后的模型决策边界
print(model)
plot_model(X, Y, model)

从结果可以看出：**线性模型准确率最高只能到 50% 左右**，对螺旋这种复杂分布，线性模型难以准确分类。

### 3. 构建两层神经网络分类（加入 ReLU 激活函数）

In [ ]:
learning_rate = 1e-3
lambda_l2 = 1e-5

# 与线性模型不同的是，两层之间加入 ReLU 激活函数
model = nn.Sequential(
    nn.Linear(D, H),
    nn.ReLU(),
    nn.Linear(H, C)
)
model.to(device)

criterion = torch.nn.CrossEntropyLoss()
# 使用 Adam 优化器（内置 L2 正则）
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=lambda_l2)

for t in range(1000):
    y_pred = model(X)
    loss = criterion(y_pred, Y)
    score, predicted = torch.max(y_pred, 1)
    acc = (Y == predicted).sum().float() / len(Y)
    print("[EPOCH]: %i, [LOSS]: %.6f, [ACCURACY]: %.3f" % (t, loss.item(), acc))
    display.clear_output(wait=True)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
# 绘制训练后的模型决策边界
print(model)
plot_model(X, Y, model)

## 结果对比

| 模型 | 最终损失 | 最终准确率 |
| --- | --- | --- |
| 线性模型（无激活函数） | 约 0.86 | 约 0.50 |
| 两层神经网络（ReLU） | 约 0.21 | 约 0.93 |

**结论**：加入 ReLU 激活函数后，网络具备了非线性表达能力，分类准确率显著提高。